# Phase 3: Feature EngineeringWe engineer critical temporal and macro-trend features required by the specification.

In [ ]:
import pandas as pd
import numpy as np
df = pd.read_csv('../data/processed/attendance_cleaned.csv')
df['Date'] = pd.to_datetime(df['Date'])


## 1. Temporal FeaturesAdding Week, Day of Semester, and Time of Day clustering.

In [ ]:
df['Week_Number'] = df['Date'].dt.isocalendar().week
df['Day_of_Semester'] = (df['Date'] - df['Date'].min()).dt.days

def get_time_of_day(time_str):
    if 'PM' in str(time_str) and not str(time_str).startswith('12'):
        return 'Afternoon'
    return 'Morning'
df['Time_of_Day'] = df['Start_Time'].apply(get_time_of_day)


## 2. Event Proximity (Holidays & Exams)

In [ ]:
test_weeks = df[df['Internal_Test_Week'].str.lower() == 'yes']['Week_Number'].unique()
df['Week_Before_Exam'] = df['Week_Number'].apply(lambda w: 1 if (w + 1) in test_weeks else 0)

# Proxy for days since last holiday
df['Is_Holiday_Adjacent'] = df['Holiday_Before_After'].apply(lambda x: 1 if str(x).lower() == 'yes' else 0)
df = df.sort_values(by=['Date', 'Lecture_Number']).reset_index(drop=True)
last_hol = df['Date'].min()
days_since = []
for _, row in df.iterrows():
    if row['Is_Holiday_Adjacent'] == 1: last_hol = row['Date']
    days_since.append(max(0, (row['Date'] - last_hol).days))
df['Days_Since_Last_Holiday'] = days_since


## 3. Rolling MomentumExtremely important: Using `.shift(1)` to prevent target leakage when calculating the rolling average of the previous 3 lectures.

In [ ]:
df['Rolling_Avg_3'] = df.groupby(['Subject', 'Section'])['Attendance_Percentage'].transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
global_mean = df['Attendance_Percentage'].mean()
df['Rolling_Avg_3'] = df['Rolling_Avg_3'].fillna(global_mean)


## 4. Leakage Prevention & Train/Test Split

In [ ]:
# We split chronologically to prevent future information from leaking into training.
n = len(df)
train_df = df.iloc[:int(n * 0.7)]
val_df = df.iloc[int(n * 0.7):int(n * 0.85)]
test_df = df.iloc[int(n * 0.85):]

train_df.to_csv('../data/processed/train.csv', index=False)
val_df.to_csv('../data/processed/val.csv', index=False)
test_df.to_csv('../data/processed/test.csv', index=False)
